<a href="https://colab.research.google.com/github/v-krishna07/polyglots-shorthand-sentiment/blob/main/inter_iit_csai_ps_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

youtube = load_dataset("shae2977/hinglish-youtube-sentiments-dataset", split="train")
tweets = load_dataset("Abhishek4896/hindi-english-code-mixed-tweets-sentiment", split="train")


In [ ]:
extra=load_dataset("airzipm/sentiment-dataset-en-hi-hinglish-v2")
print(extra)
print(extra.column_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 96761
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 8538
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 8538
    })
})
{'train': ['text', 'label'], 'validation': ['text', 'label'], 'test': ['text', 'label']}


In [ ]:

print("=== TWEETS DATASET ===")
print(tweets)
print(tweets.column_names)
print(tweets.features)
print(tweets[0])

print("\n=== YOUTUBE DATASET ===")
print(youtube)
print(youtube.column_names)
print(youtube.features)
print(youtube[0])

=== TWEETS DATASET ===
Dataset({
    features: ['tweet', 'sentiment'],
    num_rows: 498
})
['tweet', 'sentiment']
{'tweet': Value('string'), 'sentiment': Value('string')}
{'tweet': 'Finally exam clear kar liya, khushi ke aansu!', 'sentiment': 'positive'}

=== YOUTUBE DATASET ===
Dataset({
    features: ['video_id', 'comment', 'likes', 'sentiment'],
    num_rows: 3190
})
['video_id', 'comment', 'likes', 'sentiment']
{'video_id': Value('string'), 'comment': Value('string'), 'likes': Value('int64'), 'sentiment': Value('string')}
{'video_id': '0IZS7ucM9qQ', 'comment': 'Mam sir namak h kaise bnega matar pulao 😢😮😅', 'likes': 0, 'sentiment': 'Neutral'}


In [ ]:
LABEL2ID={"negative":0,"positive":2,"neutral":1}
def normalize_tweets(example):
  return {"text":example["tweet"],"label":LABEL2ID[example["sentiment"].strip().lower()]}

def normalize_youtube(example):
  return {"text":example["comment"],"label":LABEL2ID[example["sentiment"].strip().lower()]}

youtube_fixed=youtube.map(normalize_youtube,remove_columns=youtube.column_names)
tweets_fixed=tweets.map(normalize_tweets,remove_columns=tweets.column_names)


In [ ]:
from datasets import concatenate_datasets as cd
fused=cd([tweets_fixed,youtube_fixed])
print(fused[0])
print(fused)

{'text': 'Finally exam clear kar liya, khushi ke aansu!', 'label': 2}
Dataset({
    features: ['text', 'label'],
    num_rows: 3688
})


In [ ]:
texts = fused["text"]  # this is already a list-like column of strings
print(len(texts), texts[:3])

3688 ['Finally exam clear kar liya, khushi ke aansu!', 'Yaar kal ka match to full paisa vasool tha!', 'Yeh coffee bilkul thandi hai, bakwaas taste.']


NOW THE MAIN TOKENIZING

In [62]:
import emoji
from collections import Counter
from tokenizers import Tokenizer, normalizers, Regex, decoders
from tokenizers.models import BPE
from tokenizers.normalizers import Replace
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer

# Extracting top 50 emojis
VARIATION_SELECTOR = "\uFE0F"

def normalize_emoji(e):
  return e.replace(VARIATION_SELECTOR, "")

emoji_counter=Counter()

for text in texts:
  for e in emoji.emoji_list(text):
    emoji_counter[e["emoji"]]+=1

normalized_counter=Counter()

for e,count in emoji_counter.items():
  normalized_counter[normalize_emoji(e)]

top_emojis=[e for e,_ in normalized_counter.most_common(50)]

# Tokenizing
tokenizer= Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence([Replace(Regex(VARIATION_SELECTOR),"")])

tokenizer.pre_tokenizer=ByteLevel()

trainer =BpeTrainer(vocab_size=8000,special_tokens=["[UNK]","[CLS]","[SEP]","[PAD]","[MASK]"],
                    min_frequency=2)

# Train
tokenizer.train_from_iterator(texts,trainer)
tokenizer.add_tokens(top_emojis)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.save("hinglish_custom_tokenizer.json")

# test
sample="aap yaha aaiye..."
encoding = tokenizer.encode(sample)
print(encoding.tokens)
print(encoding.ids)
print(tokenizer.decode(encoding.ids))

['Ġaap', 'Ġyaha', 'Ġaai', 'ye', '...']
[308, 1655, 2931, 195, 262]
 aap yaha aaiye...
